# 2026-06-18 Day 6: KV Cache, Prefill, Decode

Goal: understand prefill vs decode, KV-cache shapes, memory formula, and cached-vs-uncached correctness.


## How To Use This Reading Notebook

Use this before touching implementation for the day. The goal is not to memorize the paper. The goal is to extract the model of the system you need for today's code and notes.

Workflow:

1. Read only the assigned sections.
2. Fill the mini-lecture answer in your own words.
3. Open the repo files listed in the roadmap.
4. Run the listed checks or record why they skip.
5. Write a short exit ticket before moving on.


## Assigned Reading

| Source | Exact Target | Why It Matters Today |
| --- | --- | --- |
| [Hugging Face KV cache guide](https://huggingface.co/docs/transformers/v4.52.1/kv_cache) | Default cache and cache-type comparison | Explains why key/value states are reused during generation and what cache options exist. |
| Day 2 attention notes | Score shapes and causal masking | KV cache is easiest to understand after attention shapes are clear. |


## Reading Summary

### Hugging Face KV Cache Guide

A decoder-only model generates one token at a time. Without caching, each step recomputes key and value tensors for all previous tokens. A KV cache stores those tensors so the next step computes only the newest token's K/V and appends them. The guide distinguishes cache implementations such as dynamic, static, offloaded, and quantized caches.

Key takeaway for implementation: the cache is an inference optimization. It should change speed and memory behavior, not model logits beyond numerical tolerance.

### Day 2 Attention Connection

During prefill, the prompt has length `T`, so scores look like `(B, H, T, T)`. During one-token cached decode, the new query length is `1` and the key/value length is the cache length, so scores look like `(B, H, 1, T_cache)`.

Key takeaway for implementation: KV cache reduces repeated computation but does not remove attention over previous tokens.


## Diagram Anchor

![Prefill decode timeline](../../assets/figures/prefill_decode_timeline.svg)

What to notice: prefill happens once, decode repeats once per generated token.

![KV cache layout](../../assets/figures/kv_cache_layout.svg)

What to notice: the cache stores both K and V for every layer.

![Cached versus uncached decode](../../assets/figures/cached_vs_uncached_decode.svg)

What to notice: correctness compares logits, not sampled text.


## Repo Connection

Open these files after reading:

| File | What To Find |
| --- | --- |
| `src/moe_transformer_lab/trainable/model.py` | Current attention forward, block forward, model forward, generation loop. |
| `configs/tiny_inference.json` | Planned inference settings and `use_kv_cache` placeholder. |
| `reports/transformer_inference_report.md` | Where later cache benchmark results should be reported. |


## Cache Shape Summary

| Tensor | Shape | Role |
| --- | --- | --- |
| `past_key[layer]` | `(B, H, T_cache, Dh)` | cached keys for one layer |
| `past_value[layer]` | `(B, H, T_cache, Dh)` | cached values for one layer |
| `new_key` | `(B, H, 1, Dh)` | newest token key |
| `new_value` | `(B, H, 1, Dh)` | newest token value |
| updated cache | `(B, H, T_cache + 1, Dh)` | append on sequence axis |

Memory formula:

```text
bytes = layers * 2 * batch * heads * seq_len * head_dim * dtype_bytes
```


## Mini-Lecture Answer Seed

Prefill processes the prompt and creates the initial cache. Decode processes one newly generated token at a time. Each layer stores K and V tensors for previous tokens. The cache memory grows linearly with batch, layers, heads, sequence length, head dimension, and dtype bytes. The factor of two appears because both keys and values are stored.


## Active Recall

1. Why does KV cache include both K and V?
2. What axis grows during decode?
3. Why compare logits in eval mode?
4. Why is cached decode not constant-time with respect to context length?

## Exit Ticket

Write a numeric KV-cache memory example using the tiny config.
